# prayoga — Reproduce F1: refusal is a single direction (GPU)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SharathSPhD/prayoga/blob/main/notebooks/01_reproduce_refusal_direction.ipynb)

Reproduces the program's linchpin (**F1**) with the *actual* prayoga code on a Colab GPU: extract the difference-in-means refusal direction in Gemma-2-2b, then **ablate** it (suppress refusal → harmful compliance) and **add** it (induce over-refusal on benign prompts).

Needs a **GPU runtime** (`Runtime > Change runtime type > T4 GPU`) and a Hugging Face token with Gemma access (`from google.colab import userdata` → set `HF_TOKEN` in the secrets panel).

⚠️ *Dual-use:* this produces an abliteration vector. Keep it for interpretability/safety research; do not redistribute the vector or the harmful generations.

In [ ]:
!git clone -q https://github.com/SharathSPhD/prayoga.git
%cd prayoga
!pip install -q -e . 2>/dev/null
import numpy as np, os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # gated Gemma access

In [ ]:
from prayoga.lm.hf_model import HFModel
from prayoga.axis_a.direction_extraction import directions_all_layers
from prayoga.axis_a.intervention_engine import InterventionEngine
from prayoga.axis_a.refusal import asr, refusal_rate
load = lambda p: [l.strip() for l in open(p) if l.strip()]
harmful, harmless = load('data/prompts/harmful.txt'), load('data/prompts/harmless.txt')
model = HFModel('google/gemma-2-2b-it'); eng = InterventionEngine(model)
print('loaded', model.n_layers, 'layers, d =', model.d_model)

### Extract the refusal direction (layer 7) and run the causal gates

In [ ]:
L = 7
d = directions_all_layers(model, harmful[:24], harmless[:24])[L]
base = asr(eng.baseline_generate(harmful[24:], 32))
abl  = asr(eng.ablate_generate(harmful[24:], d, 32))
h = model.capture_all_layers_last_token(harmful[:24])[L]; s = model.capture_all_layers_last_token(harmless[:24])[L]
sep = float(np.linalg.norm(h.mean(0)-s.mean(0)))
over = refusal_rate(eng.add_generate(harmless[24:], d, 8*sep, L, 32))
print(f'baseline harmful ASR     : {base:.2f}')
print(f'ABLATION  → harmful ASR  : {abl:.2f}   (jailbreak via direction removal)')
print(f'ADDITION  → over-refusal : {over:.2f}   (benign prompts now refused)')

You should see baseline ASR ≈ 0, **ablation ASR ≈ 0.9** (refusal removed), and **over-refusal ≈ 1.0** (benign prompts refused) — F1 reproduced. The program's other findings (dose-response F2, dimensionality F8/F18, SAE F13, symmetry F11) live in `prayoga.axis_a` / `prayoga.axis_b`; see the repo READMEs and `docs/FINDINGS.md`.